# C2.2 · Model-layer research

**Function C — Offensive Security & Research → The Security Researcher**  ·  *Security of AI*

---

**Risk.** Model cards read credulously.

**Control.** Adversarial robustness, jailbreak taxonomy, refusal analysis, capability elicitation.

**This lab.** Chart jailbreak taxonomy differences across three open-weight families.

| | |
|---|---|
| Open-source tooling | garak |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C2.2"))

Model-layer research means treating the model as the object of study with a rate, not a demo with a screenshot.

In [ ]:
from cybercommons import research

# Three techniques, each modelled as a probability of landing on a given attempt.
TECHNIQUES = {"direct override": 0.05, "context reframe": 0.35, "task nesting": 0.62}
for name, p in TECHNIQUES.items():
    r = research.trial(lambda rng, p=p: rng.random() < p, n=200, seed=11)
    print(f"{name:18s} rate {r['rate']:.3f}  ci95 {r['ci95']}  → {r['verdict']}")

Note what the middle row does to a naive claim. 'It worked' is true for all three; only one is reproducible, and the interval tells you how much of a mitigation's improvement would be noise.

In [ ]:
before = research.trial(lambda rng: rng.random() < 0.62, n=200, seed=11)
after  = research.trial(lambda rng: rng.random() < 0.48, n=200, seed=11)
print("before:", before["rate"], before["ci95"])
print("after :", after["rate"],  after["ci95"])
overlap = after["ci95"][1] >= before["ci95"][0]
print("\nintervals overlap?", overlap,
      "→", "not a demonstrated improvement" if overlap else "improvement holds")

### Expect

Direct override is not reproduced, context reframe is flaky, task nesting is reproducible. The before/after comparison shows whether the confidence intervals overlap.

### Your turn

Run 20 trials instead of 200 and watch the interval widen until the comparison says nothing. That width is why single-run jailbreak claims are unfalsifiable.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C2.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*